# Dynamic Table の CUSTOM_INCREMENTAL モード 検証ノートブック

## このノートブックについて

Zenn 記事「[Dynamic TableでMERGEロジックを自分で書く——CUSTOM_INCREMENTALが拓く複雑変換の新境地](https://zenn.dev/gtk0326/articles/i47-feature-update-2026-05-26-custom-incremental)」のハンズオン検証コードです。

記事と同じ手順を自分の Snowflake 環境で再現できます。

### 概要

`REFRESH_MODE = AUTO` で Dynamic Table を作成したとき、クエリの内容に応じて
`FULL` / `INCREMENTAL` / `CUSTOM_INCREMENTAL` のどのモードに解決されるかを確認します。

### 前提条件

- Snowflake アカウントへのアクセス権限
- `CREATE DATABASE` 権限
- ウェアハウス（`compute_wh` を使用。なければ名前を変更してください）

### 実行時間の目安

約 5〜10 分

---

> **注意**: 各セルは上から順番に実行してください。最後のクリーンアップセルを必ず実行し、検証用オブジェクトを削除してください。

## ステップ1: スキーマとソーステーブルを準備する

検証用のデータベース・スキーマを作成し、ソーステーブルと初期データを用意します。
また、`CUSTOM_INCREMENTAL` で使用するストリームも作成します。

In [ ]:
CREATE OR REPLACE DATABASE dt_custom_demo;
CREATE OR REPLACE SCHEMA dt_custom_demo.main;
USE SCHEMA dt_custom_demo.main;

-- 売上ログ（ソーステーブル）
CREATE OR REPLACE TABLE sales_log (
  sale_id  INT,
  user_id  INT,
  amount   NUMBER(10,2),
  sold_at  TIMESTAMP
);

-- sales_log 用ストリーム（CUSTOM_INCREMENTAL で使用）
CREATE OR REPLACE STREAM sales_log_stream ON TABLE sales_log;

INSERT INTO sales_log VALUES
  (1, 1, 1000.00, CURRENT_TIMESTAMP()),
  (2, 2,  500.00, CURRENT_TIMESTAMP()),
  (3, 1, 2500.00, CURRENT_TIMESTAMP()),
  (4, 3,  800.00, CURRENT_TIMESTAMP());

## ステップ2: 集計クエリの Dynamic Table を作成する（→ FULL に解決）

`SUM` + `GROUP BY` のような集計クエリは差分行だけで正しい集計値を維持できないため、
Snowflake は `FULL` を選択しリフレッシュのたびに全件を再計算します。

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE user_sales_summary
  TARGET_LAG = '1 minute'
  WAREHOUSE = compute_wh
  REFRESH_MODE = AUTO
AS
  SELECT
    user_id,
    SUM(amount)  AS total_amount,
    COUNT(*)     AS sale_count
  FROM sales_log
  GROUP BY user_id;

## ステップ3: フィルタークエリの Dynamic Table を作成する（→ INCREMENTAL に解決）

`WHERE` のみのシンプルなフィルタークエリは、差分行に同じ条件を適用するだけで結果を
維持できます。Snowflake は `INCREMENTAL` を選択し変更行のみを処理します。

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE large_sales
  TARGET_LAG = '1 minute'
  WAREHOUSE = compute_wh
  REFRESH_MODE = AUTO
AS
  SELECT sale_id, user_id, amount, sold_at
  FROM sales_log
  WHERE amount >= 1000;

## ステップ4: REFRESH USING を持つ Dynamic Table を作成する（→ CUSTOM_INCREMENTAL に解決）

`REFRESH USING` 句を追加すると、AUTO モードはユーザーが増分ロジックを明示的に定義した
と判断し `CUSTOM_INCREMENTAL` を選択します。
ここではストリームから INSERT のみを取り込む追記型パターンを実装します。

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE enriched_sales
  TARGET_LAG = '1 minute'
  WAREHOUSE = compute_wh
  REFRESH_MODE = AUTO      -- REFRESH USING があれば CUSTOM_INCREMENTAL に自動解決
  INITIALIZE = ON_CREATE
AS
  SELECT sale_id, user_id, amount, sold_at
  FROM sales_log
REFRESH USING (
  INSERT INTO SELF
    SELECT sale_id, user_id, amount, sold_at
    FROM sales_log_stream
    WHERE METADATA$ACTION = 'INSERT'
);

## ステップ5: SHOW DYNAMIC TABLES で3モードを一覧確認する

同じ `REFRESH_MODE = AUTO` で作成したのに、クエリ構造に応じて3つの異なるモードに
解決されていることを確認します。

**期待する結果:**
```
USER_SALES_SUMMARY → FULL
LARGE_SALES        → INCREMENTAL
ENRICHED_SALES     → CUSTOM_INCREMENTAL
```

In [ ]:
SHOW DYNAMIC TABLES IN SCHEMA dt_custom_demo.main;

## ステップ6: 新データを追加してインクリメンタルリフレッシュを確認する

2件のデータを追加し、INCREMENTAL モードの `large_sales` が
`amount >= 1000` の条件を差分処理で正しく適用することを確認します。

**期待する結果:** sale_id = 5 のみ取り込まれ、sale_id = 6（300円）は除外される

In [ ]:
-- 2件追加（1件は amount < 1000）
INSERT INTO sales_log VALUES
  (5, 2, 1500.00, CURRENT_TIMESTAMP()),
  (6, 3,  300.00, CURRENT_TIMESTAMP());

ALTER DYNAMIC TABLE enriched_sales REFRESH;

-- amount >= 1000 の行のみが存在するか確認
SELECT * FROM large_sales ORDER BY sale_id;

## クリーンアップ

検証で作成したオブジェクトをすべて削除します。

> **必ず実行してください。** Dynamic Table をそのままにするとバックグラウンドで
> リフレッシュが継続しクレジットが消費されます。

In [ ]:
-- Dynamic Table を停止してから削除
ALTER DYNAMIC TABLE dt_custom_demo.main.enriched_sales     SUSPEND;
ALTER DYNAMIC TABLE dt_custom_demo.main.large_sales        SUSPEND;
ALTER DYNAMIC TABLE dt_custom_demo.main.user_sales_summary SUSPEND;

-- データベースごと削除（スキーマ・テーブル・ストリームもすべて削除される）
DROP DATABASE IF EXISTS dt_custom_demo;